# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library, utilizing its Croissant metadata and robust field structure.

### Dataset Source
The dataset is defined by the Croissant schema URL below.

**URL**: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Specify the Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset and its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an mlcroissant.DatasetMetadata object

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List available record sets and their fields by their `@id`. This provides a map of the dataset for subsequent steps.


In [ ]:
# List record sets with their @ids and fields
print("Available record sets and their fields:\n")
record_sets = metadata.record_sets
for rs in record_sets:
    record_set_id = rs.id
    print(f"- RecordSet @id: {record_set_id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '-'}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '-'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}  |  Name: {field.name}")
    print()

## 3. Data Extraction
Load all fields for a selected record set into a DataFrame. (Replace variables below with discovered `@id`s from the previous step.)
For demonstration, we select the main record set that contains protein measurements (usually named similarly to `ProteinData` or as explored above).

In [ ]:
# Select a record set for extraction
# Replace the following with the actual @id for the main protein data record set, e.g., 'cr:recordSet/0' or as listed above.
main_record_set_id = metadata.record_sets[0].id  # Edit if needed, e.g., "senscience:ProteinRecordSet"
dataframes = {}

records = list(dataset.records(record_set=main_record_set_id))
main_df = pd.DataFrame(records)
dataframes[main_record_set_id] = main_df

print(f"Main record set @id: {main_record_set_id}")
print("Columns in the DataFrame (@id as key):\n", main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Process numeric fields (such as molecular weight or abundance) and group by categorical fields as available in the chosen record set.
- Filtering: Remove proteins with molecular weight (MW) below a threshold.
- Normalization: Normalize MW column.
- Grouping: By a categorical field if available (e.g., 'SampleType' or 'Description').

In [ ]:
# Choose a numeric field and a group field for EDA (adjust field ids as above)
numeric_field = None
group_field = None

# Attempt to identify typical numeric and group fields
for col in main_df.columns:
    if numeric_field is None and main_df[col].dtype in [float, int, "float64", "int64"]:
        numeric_field = col
    if group_field is None and main_df[col].dtype == object:
        group_field = col

if numeric_field is None:
    raise ValueError("No numeric field detected. Please specify the appropriate field @id from the previous overview.")
if group_field is None:
    raise ValueError("No group field detected. Please specify the appropriate field @id from the previous overview.")

print(f"Numeric field selected: {numeric_field}")
print(f"Group field selected: {group_field}\n")

# Ensure numeric_field is numeric (try to coerce if needed)
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')

threshold = main_df[numeric_field].mean()  # Use mean as a demonstration threshold

filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field if present
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (showing mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot of normalized numeric field against group_field (if group_field is present)
if group_field in filtered_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=filtered_df, x=group_field, y=f"{numeric_field}_normalized")
    plt.title(f"Normalized {numeric_field} grouped by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- Using the `mlcroissant` library and Croissant schema, we've loaded and explored the FAIR^2 dataset of protein abundances from mass spectrometry experiments.
- Data fields and record structures were referenced by their `@id` for reproducible and robust analysis.
- We've demonstrated record filtering, normalization, grouping, and visualizations to help guide scientific insight into extracellular vesicle proteomics.

**Next steps:** Detailed statistical analysis, biomarker discovery, and cross-referencing with external protein databases.